# 03.1 Sequence Basics / 序列基础

这一节的目标是把序列数据 / sequence data 的最基本表示搞清楚。  
The goal of this notebook is to make the most basic representation of sequence data completely clear.

进入 NLP 或时间序列之前，你至少要先理解：  
Before entering NLP or time-series modeling, you should at least understand:

- token / 词元或离散符号
- token id / 词元编号
- vocabulary / 词表
- sequence length / 序列长度
- embedding / 嵌入向量
- padding / 填充
- mask / 掩码

如果这些概念是模糊的，后面的 `LSTM` 和 `Attention` 很容易看晕。  
If these concepts are fuzzy, later `LSTM` and `Attention` notebooks will quickly become confusing.

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 理解离散 token 如何变成整数 id / Understand how discrete tokens become integer ids.
2. 理解序列 batch 的常见 shape / Understand common shapes for sequence batches.
3. 使用 `nn.Embedding` 把 token id 映射成 embedding / Use `nn.Embedding` to map token ids to embeddings.
4. 理解为什么需要 padding / Understand why padding is needed.
5. 构造 padding mask / Build a padding mask.
6. 为后续 `LSTM` 和 `Attention` 做 shape 准备 / Prepare the shapes needed for later `LSTM` and `Attention` notebooks.

In [ ]:
import torch
import torch.nn as nn

## 1. Token、词表与 id 映射
## Tokens, Vocabulary, and Id Mapping

最开始的序列常常是字符串或离散符号。  
A sequence often begins as strings or other discrete symbols.

模型不能直接吃字符串，所以通常要先映射到整数 id。  
A model cannot directly consume strings, so we usually map them to integer ids first.

In [ ]:
sentences = [
    ["i", "like", "pytorch"],
    ["you", "like", "deep", "learning"],
    ["i", "study"],
]

special_tokens = ["<pad>", "<unk>"]
vocab = special_tokens + sorted({token for sent in sentences for token in sent})
stoi = {token: idx for idx, token in enumerate(vocab)}
itos = {idx: token for token, idx in stoi.items()}

print("vocab =", vocab)
print("stoi =", stoi)
print("itos[2] =", itos[2])

In [ ]:
encoded_sentences = [[stoi.get(token, stoi["<unk>"]) for token in sent] for sent in sentences]

print("original sentences / 原始句子:", sentences)
print("encoded sentences / 编码后句子:", encoded_sentences)

## 2. 序列长度 / Sequence Length

序列模型里最常见的一个维度就是 `sequence length`。  
One of the most common dimensions in sequence modeling is the sequence length.

例如 / For example:

- `['i', 'like', 'pytorch']` 的长度是 3
- `['you', 'like', 'deep', 'learning']` 的长度是 4

问题是：同一个 batch 里的序列长度经常不一样。  
The problem is that sequence lengths within the same batch are often different.

## 3. 为什么需要 padding / Why Do We Need Padding?

因为张量要求一个 batch 内部形状统一，所以需要把短序列补到相同长度。  
Because a tensor requires a uniform shape within a batch, shorter sequences must be padded to the same length.

通常用 `<pad>` token 填充。  
We usually pad with the `<pad>` token.

In [ ]:
pad_id = stoi["<pad>"]
max_len = max(len(seq) for seq in encoded_sentences)
padded_sentences = [seq + [pad_id] * (max_len - len(seq)) for seq in encoded_sentences]

batch_ids = torch.tensor(padded_sentences, dtype=torch.long)

print("max_len =", max_len)
print("padded_sentences =", padded_sentences)
print("batch_ids =\n", batch_ids)
print("batch_ids.shape =", batch_ids.shape)

这里 `batch_ids.shape == (3, 4)` 表示：  
`batch_ids.shape == (3, 4)` means:

- `3` 个样本 / 3 samples in the batch
- 每个样本补到了长度 `4` / each sample is padded to length 4

在很多 PyTorch 序列模型里，我们常用 `batch_first=True`，所以 shape 写成：  
In many PyTorch sequence models, we often use `batch_first=True`, so the shape is written as:

- `(batch_size, seq_len)`
- `(batch_size, seq_len, embedding_dim)`

In [ ]:
# 练习 1 / Exercise 1
# 给定下面三条序列 id，请把它们 padding 到相同长度。
# Given the three sequences below, pad them to the same length.

seqs = [[3, 5], [2, 4, 6, 7], [9]]
pad_id = 0

# max_len =
# padded =
# batch =
# print(batch)
# print(batch.shape)

In [ ]:
# 练习 1 参考答案 / Exercise 1 Reference Solution

seqs = [[3, 5], [2, 4, 6, 7], [9]]
pad_id = 0
max_len = max(len(seq) for seq in seqs)
padded = [seq + [pad_id] * (max_len - len(seq)) for seq in seqs]
batch = torch.tensor(padded, dtype=torch.long)
print(batch)
print(batch.shape)

## 4. `nn.Embedding` / `nn.Embedding`

token id 本身只是编号，不包含语义。  
A token id is only an index; it does not carry semantic meaning by itself.

`Embedding` 的作用是把每个 token id 映射成一个稠密向量。  
The role of an embedding is to map each token id to a dense vector.

In [ ]:
vocab_size = len(vocab)
embedding_dim = 6
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

embedded = embedding(batch_ids)

print("batch_ids.shape =", batch_ids.shape)
print("embedded.shape =", embedded.shape)

这里的 shape 从 `(batch_size, seq_len)` 变成了 `(batch_size, seq_len, embedding_dim)`。  
Here the shape changes from `(batch_size, seq_len)` to `(batch_size, seq_len, embedding_dim)`.

也就是说，每个 token 都变成了一个向量。  
That means every token becomes a vector.

In [ ]:
print("batch_ids[0] =", batch_ids[0])
print("embedded[0].shape =", embedded[0].shape)
print("embedded[0] =\n", embedded[0])

In [ ]:
# 练习 2 / Exercise 2
# 构造一个 embedding 层，词表大小 20，embedding 维度 8。
# Build an embedding layer with vocab size 20 and embedding dimension 8.
#
# 然后把一个 shape 为 (4, 5) 的 token id batch 喂进去，打印输出 shape。
# Then feed it a token-id batch of shape (4, 5) and print the output shape.

# emb =
# batch =
# out =
# print(out.shape)

In [ ]:
# 练习 2 参考答案 / Exercise 2 Reference Solution

emb = nn.Embedding(20, 8)
batch = torch.randint(low=0, high=20, size=(4, 5))
out = emb(batch)
print(out.shape)

## 5. Padding Mask / Padding Mask

虽然 padding 让 batch 形状统一了，但 `<pad>` 并不是真实内容。  
Although padding makes the batch shape uniform, `<pad>` is not real content.

所以很多序列模型需要知道：哪些位置是 padding。  
So many sequence models need to know which positions are padding.

这就是 `padding mask` 的作用。  
That is the role of a padding mask.

In [ ]:
padding_mask = batch_ids == pad_id

print("batch_ids =\n", batch_ids)
print("padding_mask =\n", padding_mask)
print("padding_mask.shape =", padding_mask.shape)

这里 `True` 表示当前位置是 padding。  
Here `True` means that the position is padding.

这类 mask 在后面的 `Attention` notebook 里会非常重要。  
This kind of mask becomes very important in the later `Attention` notebook.

In [ ]:
# 练习 3 / Exercise 3
# 给定下面的 batch ids，构造 padding mask。
# Given the batch ids below, construct the padding mask.

batch = torch.tensor([
    [4, 5, 0, 0],
    [1, 2, 3, 0],
    [7, 8, 9, 1],
])
pad_id = 0

# mask =
# print(mask)

In [ ]:
# 练习 3 参考答案 / Exercise 3 Reference Solution

batch = torch.tensor([
    [4, 5, 0, 0],
    [1, 2, 3, 0],
    [7, 8, 9, 1],
])
pad_id = 0
mask = batch == pad_id
print(mask)

## 6. 常见 shape 对照表 / Common Shape Reference Table

你应该尽快习惯下面这组 shape：  
You should get comfortable with this set of shapes as early as possible:

- token ids: `(batch_size, seq_len)`
- embeddings: `(batch_size, seq_len, embedding_dim)`
- padding mask: `(batch_size, seq_len)`

后面只是在这些基础上增加新的维度语义。  
Later notebooks mostly add new semantics on top of these basic dimensions.

## 7. 小结 / Summary

这一节的核心不是 API 数量，而是把序列输入格式彻底想清楚。  
The core of this notebook is not the number of APIs, but fully understanding the input format of sequence data.

你现在应该能回答 / You should now be able to answer:

1. 为什么 token 常要先映射成整数 id？ / Why are tokens often mapped to integer ids first?
2. 为什么同一个 batch 里需要 padding？ / Why do we need padding within the same batch?
3. `Embedding` 为什么会把 shape 从 `(B, T)` 变成 `(B, T, D)`？ / Why does an `Embedding` turn `(B, T)` into `(B, T, D)`?
4. `padding mask` 的作用是什么？ / What is the role of a padding mask?

下一步建议 / Suggested next step:

- 进入 `LSTM` notebook，观察这些序列张量是如何被循环模型处理的 / Move to the `LSTM` notebook and see how these sequence tensors are processed by a recurrent model.